# Concept Probe — comparison-pair expansion (CPU; no new model inference)

**Goal:** Find a broader pool of approximately 20 *distinct target concepts* for which X-vs-Y Probe discrimination is reproducible. This notebook is a **screening experiment**, not a confirmatory test or a test of suppression/semantic distance.

- Reuses **all 40 previously extracted concepts and 5,200 hidden-state records** from the completed Qwen2.5-0.5B-Instruct experiment.
- Changes the comparison concept Y within predefined topic groups. It creates **new X/Y pairs**, not new semantic-description families or new model forward passes.
- Screens two non-identical comparators per target where possible and retains original comparator pairs. The true same X/Y pair reversed is **not** treated as independent evidence.
- Uses the same family-level ROC-AUC protocol (six training-family folds; five unseen evaluation families), primary readout: layer 24, `assistant_start`, neutral instruction.
- Optional complete 252-label balanced family permutations for every shortlisted **unique unordered** pair.
- A shortlisted pair is **only a candidate**. To validate any final 20 concepts, rerun on **fresh descriptions** and check concept-versus-comparator robustness before the thesis's semantic-distance study.

## How to run
1. Open this file in Colab, with ordinary **CPU runtime** (GPU is unnecessary).
2. Set a fine-grained GitHub token in **Colab Secrets** named `GITHUB_RESULTS_TOKEN` with Contents read/write access only to `ryouy/GraduationThesis`.
3. Run all cells. The input checkpoint ZIPs are already in GitHub, so existing measurements are not overwritten. Screening and results are saved in a separate experiment directory.


In [ ]:
%pip -q install "scikit-learn==1.6.1" pandas numpy matplotlib tqdm requests threadpoolctl tabulate


In [ ]:
import io, os, json, re, hashlib, math, itertools, runpy, zipfile, time
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import quote
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from threadpoolctl import threadpool_limits
from google.colab import userdata

SEED = 42
C = 0.1
LAYERS = [22, 23, 24]
POSITIONS = ["context_end", "assistant_start"]
CONDITIONS = ["neutral", "mention_x", "think_x", "avoid_x", "think_y"]
BATCH_SIZE = 64
OUT = Path('/content/qwen_probe_pair_expansion')
OUT.mkdir(parents=True, exist_ok=True)
CACHE = OUT / 'checkpoint_batches'
CACHE.mkdir(exist_ok=True)
GITHUB_REPOSITORY = 'ryouy/GraduationThesis'
SOURCE_PREFIX = '1110_qwen_probe_20pairs/results/20261110_qwen20pairs_v1/checkpoint_chunks'
EXPERIMENT_SLUG = '1111_qwen_probe_pair_expansion'
RUN_ID = '20261111_comparator_screen_v1'
DEST_PREFIX = f'{EXPERIMENT_SLUG}/results/{RUN_ID}'
GITHUB_BRANCH = 'main'
try:
    GITHUB_TOKEN = userdata.get('GITHUB_RESULTS_TOKEN')
except Exception as exc:
    raise RuntimeError('Add GITHUB_RESULTS_TOKEN to Colab Secrets and enable notebook access.') from exc
assert isinstance(GITHUB_TOKEN, str) and GITHUB_TOKEN.strip(), 'GITHUB_RESULTS_TOKEN is missing'
HEADERS = {'Authorization': 'Bearer ' + GITHUB_TOKEN.strip(), 'Accept':'application/vnd.github+json', 'X-GitHub-Api-Version':'2022-11-28'}

def url_for(path):
    return f'https://api.github.com/repos/{GITHUB_REPOSITORY}/contents/{quote(path,safe="/")}'

def blob_sha(content):
    return hashlib.sha1(f'blob {len(content)}\0'.encode() + content).hexdigest()

def github_read(path):
    r = requests.get(url_for(path), headers=HEADERS, timeout=60)
    if r.status_code == 404: return None
    r.raise_for_status()
    info = r.json()
    raw = requests.get(info['download_url'], timeout=180)
    raw.raise_for_status()
    assert blob_sha(raw.content) == info['sha'], f'GitHub blob hash mismatch: {path}'
    return raw.content

def github_publish(path, content):
    """Update only experiment-owned result files, with GitHub SHA concurrency protection."""
    assert path.startswith(DEST_PREFIX + '/'), 'Refusing writes outside run-specific results folder'
    r = requests.get(url_for(path), headers=HEADERS, timeout=60)
    if r.status_code not in (200,404): r.raise_for_status()
    existing = None if r.status_code == 404 else r.json()
    if existing and existing['sha'] == blob_sha(content):
        print('Already up to date:', path)
        return
    import base64
    payload = {'message': 'Save Qwen Probe comparator expansion: ' + path.rsplit('/',1)[-1], 'content': base64.b64encode(content).decode('ascii'), 'branch': GITHUB_BRANCH}
    if existing: payload['sha'] = existing['sha']
    result = requests.put(url_for(path), headers=HEADERS, json=payload, timeout=180)
    result.raise_for_status()
    assert result.json()['content']['sha'] == blob_sha(content)
    print('Published:',path)

print('Experiment results:', f'https://github.com/{GITHUB_REPOSITORY}/tree/main/{DEST_PREFIX}')


## 1. Rebuild the exact original prompt order

These cells are **unchanged stimulus definitions** from the completed 20-pair notebook. The stored SHA256 of the source stimulus bank and the ordered prompt list must match the original extraction; otherwise all calculations stop.


In [ ]:
import requests

SOURCE_URL = "https://raw.githubusercontent.com/ryouy/GraduationThesis/main/1104_instruction_probe_20pairs/prepare.py"
EXPECTED_PREPARE_SHA256 = "e7b9424f02db0058824aaf742ace6d53ac472dfadad1065ff9ec1f42c33a718b"

source_response = requests.get(SOURCE_URL, timeout=60)
source_response.raise_for_status()
prepare_bytes = source_response.content
actual_sha = hashlib.sha256(prepare_bytes).hexdigest()
assert actual_sha == EXPECTED_PREPARE_SHA256, (
    f"Frozen training source changed: expected {EXPECTED_PREPARE_SHA256}, got {actual_sha}"
)

prepare_path = OUT / "frozen_prepare.py"
prepare_path.write_bytes(prepare_bytes)
source_ns = runpy.run_path(str(prepare_path))

PAIRS = dict(source_ns["PAIRS"])
TRAIN_FACTS = dict(source_ns["FACTS"])
TRAIN_FRAMES = list(source_ns["TRAIN_FRAMES"])

assert len(PAIRS) == 20
assert len(TRAIN_FACTS) == 40
assert all(len(TRAIN_FACTS[c]) == 11 for c in TRAIN_FACTS)
assert len(TRAIN_FRAMES) == 5

print("Frozen training source verified.")
print("Pairs:", len(PAIRS), "| Concepts:", len(TRAIN_FACTS), "| Train families/concept:", 6)
display(pd.DataFrame([{"X": x, "Y": y} for x, y in PAIRS.items()]))


In [ ]:
HELDOUT = {
    "banana": [
        "The crop is cut while still green and ripened later in temperature-controlled rooms.",
        "The plant's broad leaves often split lengthwise after strong tropical winds.",
        "A plantation worker removed a large hanging flower structure before the next harvest.",
        "The fruit's natural starch gradually changes into sugars during storage.",
        "A smooth puree made from the produce is commonly used as an early infant food."
    ],
    "apple": [
        "Bees visited the orchard during the short period when its trees were in blossom.",
        "Growers grafted a preferred variety onto dwarfing rootstock to control tree height.",
        "The harvest was kept in a low-oxygen storage room so it would remain usable through winter.",
        "A preserve maker relied on the fruit's natural pectin to help a mixture set.",
        "The orchard owner thinned young fruit so the remaining crop would develop more evenly."
    ],
    "elephant": [
        "The matriarch guided several generations along a seasonal route between feeding grounds.",
        "The herd gathered around a newborn calf while the adults remained close together.",
        "Low-frequency rumbles helped distant members of the family group remain in contact.",
        "The animals dug into a dry riverbed to reach water during the drought.",
        "A large herbivore changed the woodland by pushing over small trees and opening clearings."
    ],
    "giraffe": [
        "A newborn was able to stand and walk shortly after being delivered.",
        "Males sometimes compete by swinging their necks against one another.",
        "The animal sleeps for brief periods, often remaining upright.",
        "A mother gave birth while standing, and the newborn dropped to the ground below.",
        "Several individuals browsed among scattered acacia trees across the open savanna."
    ],
    "piano": [
        "The accompanist followed a singer's breathing during a classical art-song recital.",
        "A jazz trio used the instrument to provide the harmonic foundation between bass and drums.",
        "The student practiced separate hand parts before combining them for the examination.",
        "A composer wrote a multi-movement concerto for a solo performer and orchestra.",
        "The stage crew moved the large instrument on a wheeled platform before the concert."
    ],
    "violin": [
        "The concertmaster gave the orchestra its tuning pitch before the conductor entered.",
        "The first chair led the string section through a difficult passage.",
        "The soloist shaped a sustained note with a controlled change in pitch and intensity.",
        "A folk musician played a lively melody during a village dance.",
        "The quartet's upper voice carried the melody while the other players supplied harmony."
    ],
    "cat": [
        "The pet rubbed its cheeks against the furniture to leave a familiar scent.",
        "A nocturnal hunter watched a small movement from the edge of a windowsill.",
        "The animal pressed its front paws alternately into a soft blanket.",
        "Its whiskers helped it judge whether a narrow opening was wide enough to pass through.",
        "The household pet landed upright after jumping down from a high shelf."
    ],
    "dog": [
        "A rescue team followed the animal's scent trail through the rubble after the earthquake.",
        "The companion greeted its owner by spinning excitedly near the doorway.",
        "A shepherd guided a flock across the field without allowing the animals to scatter.",
        "The trained animal detected a hidden substance inside a piece of luggage.",
        "A guide companion helped a pedestrian navigate a busy crossing."
    ],
    "lion": [
        "Females cooperated during a coordinated hunt across the grassland.",
        "The cubs played near a sheltered den while the adults rested nearby.",
        "A deep roar carried across the territory shortly after sunset.",
        "Related females remained together as the core of the social group.",
        "The predator rested in the shade during the hottest part of the day."
    ],
    "tiger": [
        "A solitary hunter dragged its prey into dense cover before feeding.",
        "The animal marked the edge of its range with scent and scratches on tree trunks.",
        "A camera trap recorded one large cat moving alone through a snowy forest.",
        "The predator crossed a river while pursuing prey along the opposite bank.",
        "A conservation team monitored a wide forest territory used by a single individual."
    ],
    "dolphin": [
        "A mother guided her calf through shallow water while it continued to nurse.",
        "Several animals cooperated to herd a school of fish into a compact group.",
        "The marine mammal rested while allowing one half of its brain to remain alert.",
        "A playful animal repeatedly leapt beside the bow of a moving vessel.",
        "Researchers identified individuals by the distinctive patterns of their social calls."
    ],
    "shark": [
        "The predator detected weak electrical signals produced by nearby muscle movements.",
        "A species deposited leathery egg cases that became attached to underwater vegetation.",
        "Sensory structures along the body detected small movements in the surrounding water.",
        "The animal's body coloring made it difficult to distinguish from the water above and below.",
        "A marine predator approached prey by following chemical traces carried through the current."
    ],
    "eagle": [
        "The bird circled above a mountain valley while searching the ground for food.",
        "A breeding pair returned to the same cliff ledge to raise its young.",
        "The raptor detected a small animal from high above an open field.",
        "The nest became larger each breeding season as new branches were added.",
        "The hunter descended rapidly toward prey after watching it from a distant perch."
    ],
    "penguin": [
        "Adults took turns keeping an egg warm during the long breeding period.",
        "A parent recognized its own chick among thousands by listening to its call.",
        "Dense layers of feathers trapped warm air while the bird remained in icy water.",
        "Large breeding colonies gathered on remote shores during the nesting season.",
        "An adult brought food back to the colony and regurgitated it for its young."
    ],
    "butterfly": [
        "The insect migrated over a great distance between seasonal breeding grounds.",
        "A pollinator transferred pollen while visiting several blossoms in the same garden.",
        "The female placed tiny eggs on a particular host plant.",
        "Large groups gathered in a sheltered grove during the cold season.",
        "The species followed a recurring route between its summer and winter habitats."
    ],
    "spider": [
        "The arachnid injected venom to immobilize an insect before feeding.",
        "A female carried a cluster of young on the underside of her body.",
        "Specialized hairs on its legs detected vibrations from nearby movement.",
        "The hunter remained motionless before suddenly ambushing an insect.",
        "As it grew, the animal periodically shed its outer covering."
    ],
    "rose": [
        "A gardener grafted a prized cultivar onto a hardy rootstock.",
        "A florist removed lower leaves before arranging the stems in a vase.",
        "After flowering, the shrub developed a small fruit that remained through autumn.",
        "A perfume maker distilled aromatic oil from the plant's blossoms.",
        "The gardener pruned the woody stems after the first flush of spring growth."
    ],
    "sunflower": [
        "Young plants turned their heads toward the sun during an early stage of growth.",
        "The crop was harvested for cooking oil and livestock feed.",
        "A field of tall annual plants formed a bright border around the farm.",
        "The heavy flower heads began to droop as the seeds matured near the end of summer.",
        "The central disk contained hundreds of tiny individual flowers."
    ],
    "carrot": [
        "The vegetable supplied beta-carotene, which the body can convert into vitamin A.",
        "The produce was preserved in vinegar and spices for use during winter.",
        "A rabbit nibbled the crisp crop from a garden bed.",
        "The plant's finely divided foliage resembled a delicate green fern.",
        "The farmer thinned the seedlings early to give the underground crop enough room to develop."
    ],
    "potato": [
        "Farmers propagated the crop by planting pieces that each contained a viable eye.",
        "The harvest was stored in a cool, dark cellar to delay sprouting.",
        "A field of small pale blossoms appeared above the underground crop.",
        "Starch extracted from the produce was used to thicken a sauce.",
        "The farmer rotated the crop to reduce the build-up of soil-borne disease."
    ],
    "lemon": [
        "The chef added a sharp acidic note to balance a rich sauce.",
        "A preserved slice was packed in salt for use in North African cooking.",
        "The aromatic oil was released when the outer rind was twisted over a dish.",
        "The drink was served with a distinctly tart finish.",
        "The tree produced fragrant white blossoms before the next harvest."
    ],
    "orange": [
        "The outer rind was dried and candied before being used in a confection.",
        "A grove produced fragrant blossoms during the spring flowering season.",
        "The snack was divided into easy-to-share sections for a group of children.",
        "The crop was harvested during the cooler months in a warm growing region.",
        "The fruit was packed into crates after being sorted by size and ripeness."
    ],
    "bread": [
        "The baker scored the dough before placing it inside a hot stone chamber.",
        "A starter culture fermented on the counter for several days before being used.",
        "The crust crackled as steam escaped from the loaf while it cooled.",
        "The dough expanded after resting in a warm place.",
        "A picnic lunch was prepared by placing several fillings between two slices."
    ],
    "rice": [
        "The crop thrived in flooded fields during the warm monsoon season.",
        "The grains were milled to remove their outer husks before cooking.",
        "A sticky variety was shaped into small cakes for a seasonal festival.",
        "The harvest was dried before being stored in large sacks.",
        "The plant grew in long clusters above shallow standing water."
    ],
    "bicycle": [
        "The rider changed gears before climbing a steep hill.",
        "A racing group traveled in a tightly organized peloton.",
        "The frame was secured to a rack outside the railway station.",
        "The athlete wore clipped shoes during a long-distance road event.",
        "A mechanic adjusted the spokes to make a wheel run straight."
    ],
    "motorcycle": [
        "The rider shifted gears with a foot lever while operating a hand clutch.",
        "The exhaust produced a low rumble while the vehicle waited at a traffic light.",
        "A touring rider packed luggage beside the rear seat before a long trip.",
        "The machine leaned sharply into a curve on a mountain road.",
        "A mechanic checked tire pressure and the tension of the drive chain."
    ],
    "train": [
        "A conductor checked tickets as passengers boarded at the platform.",
        "The locomotive hauled freight across the continent overnight.",
        "The timetable listed a departure from the central terminal.",
        "A high-speed service connected two major cities in less than an hour.",
        "A signal changed to green before the vehicle continued along its route."
    ],
    "airplane": [
        "Cabin crew demonstrated the emergency exits before departure.",
        "Passengers fastened their seat belts as the aircraft climbed.",
        "The pilot requested clearance from air traffic control.",
        "A baggage carousel delivered the travelers' luggage after arrival.",
        "The vehicle crossed an ocean in several hours."
    ],
    "boat": [
        "The crew dropped anchor in a sheltered bay and remained there overnight.",
        "A harbor pilot guided the vessel through a narrow channel.",
        "The passengers put on life jackets before leaving the dock.",
        "The craft rocked gently as waves passed beneath it.",
        "A sail caught the wind during a coastal voyage."
    ],
    "submarine": [
        "The crew remained below the surface for weeks without coming ashore.",
        "Hydrophones detected distant propeller noise in the dark water.",
        "The vessel launched a torpedo during a naval exercise.",
        "A sonar operator mapped the seabed in near darkness.",
        "The crew used an escape chamber during a rescue drill."
    ],
    "hammer": [
        "A blacksmith struck an anvil to flatten a heated bar of metal.",
        "Construction workers demolished a brick wall using repeated impacts.",
        "A courtroom official used a wooden mallet to call the session to order.",
        "The carpenter drove a wooden wedge into a tight joint.",
        "A metalworker shaped a hot piece of material through repeated blows."
    ],
    "screwdriver": [
        "An electrician tightened a terminal screw inside a wall switch.",
        "A repairer opened a small electronic casing to replace a battery.",
        "A technician adjusted a precision fastener on a camera tripod.",
        "The handle rotated while the metal tip remained seated in a narrow recess.",
        "A small tool was magnetized so that tiny fasteners would stay attached."
    ],
    "scissors": [
        "A tailor trimmed excess fabric around a paper pattern.",
        "A barber shaped a client's hair with repeated small snips.",
        "A child made paper snowflakes by folding and cutting a sheet.",
        "A gardener removed a dead flower head with a bypass cutting tool.",
        "The implement opened and closed repeatedly using one hand."
    ],
    "knife": [
        "A chef filleted a fish along its backbone.",
        "The cook sharpened the edge against a whetstone before preparing dinner.",
        "A surgeon made a precise incision during an operation.",
        "The butcher carved thin portions from a roasted joint of meat.",
        "A hiker prepared kindling with a single-edged cutting implement."
    ],
    "clock": [
        "The alarm rang at dawn after being set the previous evening.",
        "A pendulum swung steadily inside a wooden case.",
        "The station announced that the mechanism had lost three minutes.",
        "The tower chimed at the beginning of every hour.",
        "A watchmaker replaced a tiny spring inside the timekeeping mechanism."
    ],
    "calendar": [
        "The office marked public holidays and project deadlines on a yearly planner.",
        "A team arranged its milestones across the twelve months of the year.",
        "The appointment was moved from one weekday to another.",
        "The wall display showed school terms and vacation periods.",
        "The organizer crossed off each day until the wedding ceremony."
    ],
    "refrigerator": [
        "The freezer compartment formed a layer of ice around the trays.",
        "A sensor alerted the owner when the door remained open too long.",
        "The compressor cycled on and off to maintain a stable internal temperature.",
        "Leftovers were stored safely overnight instead of being left on the counter.",
        "The kitchen appliance dispensed chilled water through a small opening in its door."
    ],
    "oven": [
        "A baker preheated the chamber before placing a tray of cookies inside.",
        "The roast developed a browned crust under sustained dry heat.",
        "A thermostat controlled the temperature during a long baking process.",
        "A fan circulated hot air around the food during cooking.",
        "The cook browned vegetables under an intense upper heating element."
    ],
    "camera": [
        "A photographer adjusted exposure before capturing a fast-moving scene.",
        "A memory card stored hundreds of digital image files.",
        "A flash illuminated the subject in a dimly lit room.",
        "The device recorded a short video during the ceremony.",
        "A portrait was carefully framed while looking through a viewfinder."
    ],
    "telescope": [
        "An astronomer tracked a comet as it moved across the night sky.",
        "A large mirror collected faint light from a distant nebula.",
        "The observatory opened its dome after sunset.",
        "The instrument resolved two stars that appeared close together.",
        "A motorized mount compensated for the rotation of the Earth."
    ]
}

assert set(HELDOUT) == set(TRAIN_FACTS)
assert all(len(v) == 5 for v in HELDOUT.values())

FEATURE_AUDIT = {
    "banana/apple": "Training: peel, color, shape, taste, common food uses. Held-out: crop handling, plant growth, storage, pectin, orchard pollination and cultivar management.",
    "elephant/giraffe": "Training: trunk, tusks, ears, neck, spots and body shape. Held-out: migration, family groups, calves, vocal communication, birth and savanna behavior.",
    "piano/violin": "Training: keyboard, keys, strings, hammers and bow. Held-out: accompaniment, ensemble roles, repertoire, rehearsal and performance contexts.",
    "cat/dog": "Training: purring, claws, barking, fetching and leash walking. Held-out: scent work, territory marking, service tasks, body language and household behavior.",
    "lion/tiger": "Training: mane, stripes, prides and solitary appearance. Held-out: cooperative hunting, cub care, territory, habitat use and camera-trap observations.",
    "dolphin/shark": "Training: blowhole, echolocation, teeth, gills and body shape. Held-out: maternal care, cooperative feeding, sleep, electroreception and reproduction.",
    "eagle/penguin": "Training: flight, talons, hooked beak, flippers and flightlessness. Held-out: nesting, incubation, chick recognition, colony behavior and hunting observation.",
    "butterfly/spider": "Training: wings, caterpillar, chrysalis, silk and webs. Held-out: migration, pollination, egg placement, vibration sensing and predatory behavior.",
    "rose/sunflower": "Training: petals, fragrance, prickles, yellow flower heads and seeds. Held-out: grafting, pruning, perfume production, growth orientation and oil production.",
    "carrot/potato": "Training: orange roots, leafy tops, tubers, eyes and common cooking forms. Held-out: nutrition, preservation, crop propagation, storage and plant growth.",
    "lemon/orange": "Training: sourness, color, peel, segments and juice. Held-out: preservation, rind oils, blossoms, harvest and culinary balancing.",
    "bread/rice": "Training: flour dough, loaves, yeast, cereal grains, paddies and sushi. Held-out: baking process, fermentation, crust, milling, drying and crop cultivation.",
    "bicycle/motorcycle": "Training: pedals, chain, human power, engine and throttle. Held-out: racing, gears, clutch operation, touring, road handling and maintenance.",
    "train/airplane": "Training: rails, carriages, stations, fixed wings and runway. Held-out: ticket inspection, freight, timetables, cabin safety and air-traffic control.",
    "boat/submarine": "Training: surface travel, oars, hull, underwater travel, ballast and periscope. Held-out: anchoring, harbor navigation, life jackets, hydrophones and sonar.",
    "hammer/screwdriver": "Training: impact, head, nails, claw, fasteners and tip. Held-out: blacksmithing, demolition, courtroom mallet use, electronics repair and precision adjustment.",
    "scissors/knife": "Training: paired blades, finger loops, single blade and food cutting. Held-out: tailoring, hair cutting, paper craft, filleting, surgery and carving.",
    "clock/calendar": "Training: hands, ticking, dial, hours, dates and month grids. Held-out: alarms, pendulums, mechanical repair, project planning and appointments.",
    "refrigerator/oven": "Training: cold storage, compressor, shelves, baking chamber and roasting. Held-out: freezer behavior, door sensors, leftovers, preheating, convection and broiling.",
    "camera/telescope": "Training: image sensor, shutter, lens, photographs, stars and magnification. Held-out: exposure, memory cards, flash, viewfinder, comet tracking and observatory operation."
}

display(pd.DataFrame([{"Pair": k, "Feature separation (manual audit)": v} for k, v in FEATURE_AUDIT.items()]))


In [ ]:
EVAL_FRAMES = [
    "A field report notes: {s}",
    "An observer recorded: {s}",
    "The account states: {s}",
    "A research note describes the scene: {s}"
]

INSTRUCTIONS = {
    "neutral": "Read the context carefully.",
    "mention_x": "The topic word is {x}.",
    "think_x": "Think about {x}.",
    "avoid_x": "Do not think about {x}.",
    "think_y": "Think about {y}."
}

def make_prompt(x, y, condition, text):
    instruction = INSTRUCTIONS[condition].format(x=x, y=y)
    prefix = f"Instruction: {instruction}\n\nContext:\n"
    prompt = prefix + text + "\n\nGive one short response about the context."
    context_end_char = len(prefix) + len(text) - 1
    return prompt, context_end_char

training_rows = []
for x, y in PAIRS.items():
    for concept, label in [(x, 1), (y, 0)]:
        for family_idx, fact in enumerate(TRAIN_FACTS[concept][:6]):
            for wrapper_idx, frame in enumerate(TRAIN_FRAMES):
                text = frame.format(s=fact)
                prompt, end_char = make_prompt(x, y, "neutral", text)
                training_rows.append({
                    "pair": x, "comparison": y, "concept": concept, "label": label,
                    "cue_kind": "x" if label == 1 else "y",
                    "family_index": family_idx,
                    "family": f"{concept}_train_{family_idx}",
                    "wrapper_index": wrapper_idx,
                    "text": text, "condition": "neutral",
                    "prompt": prompt, "context_end_char": end_char
                })

evaluation_rows = []
for x, y in PAIRS.items():
    for concept, cue_kind in [(x, "x"), (y, "y")]:
        for family_idx, sentence in enumerate(HELDOUT[concept]):
            for wrapper_idx, frame in enumerate(EVAL_FRAMES):
                text = frame.format(s=sentence)
                for condition in CONDITIONS:
                    prompt, end_char = make_prompt(x, y, condition, text)
                    evaluation_rows.append({
                        "pair": x, "comparison": y, "concept": concept,
                        "label": 1 if cue_kind == "x" else 0,
                        "cue_kind": cue_kind,
                        "family_index": family_idx,
                        "family": f"{concept}_eval_{family_idx}",
                        "wrapper_index": wrapper_idx,
                        "text": text, "condition": condition,
                        "prompt": prompt, "context_end_char": end_char
                    })

train_df = pd.DataFrame(training_rows)
eval_df = pd.DataFrame(evaluation_rows)

assert len(train_df) == 40 * 6 * 5 == 1200
assert len(eval_df) == 20 * 2 * 5 * 4 * 5 == 4000
assert train_df.groupby("concept").size().eq(30).all()
assert eval_df.groupby(["pair", "condition", "cue_kind"]).size().eq(20).all()

# Name-leakage check: neither X nor Y may appear as a standalone word in held-out text.
def has_concept_name(text, concept):
    # Exact word plus common English plural suffixes.
    pattern = r"(?i)(?<![A-Za-z])" + re.escape(concept) + r"(?:s|es)?(?![A-Za-z])"
    return bool(re.search(pattern, text))

leaks = []
for row in eval_df.drop_duplicates(["pair", "family", "text"]).itertuples():
    if has_concept_name(row.text, row.concept) or has_concept_name(row.text, row.comparison):
        leaks.append((row.pair, row.concept, row.text))
assert not leaks, f"Concept-name leakage found: {leaks[:5]}"

# Exact duplicate checks across held-out families and between train/evaluation texts.
def norm_text(s):
    return re.sub(r"\s+", " ", s.strip().lower())

heldout_unique = eval_df.drop_duplicates(["concept", "family"])[["concept", "family", "text"]]
assert heldout_unique.groupby("concept").size().eq(5).all()
assert heldout_unique["text"].map(norm_text).nunique() == len(heldout_unique)
train_texts = set(train_df["text"].map(norm_text))
heldout_texts = set(heldout_unique["text"].map(norm_text))
assert not (train_texts & heldout_texts), "Exact training/evaluation text overlap found."

# No duplicated prompts within the same design.
assert not train_df["prompt"].duplicated().any()
assert not eval_df["prompt"].duplicated().any()

PROMPTS = list(dict.fromkeys(train_df["prompt"].tolist() + eval_df["prompt"].tolist()))
PROMPT_TO_INDEX = {p: i for i, p in enumerate(PROMPTS)}
PROMPT_HASH = hashlib.sha256("\n".join(PROMPTS).encode("utf-8")).hexdigest()

print("Training rows:", len(train_df))
print("Evaluation rows:", len(eval_df))
print("Unique forward prompts:", len(PROMPTS))
print("Held-out semantic families:", len(heldout_unique))
print("Concept-name leakage: none")
print("Exact train/evaluation text overlap: none")

train_df.to_csv(OUT / "training_design.csv", index=False)
eval_df.to_csv(OUT / "evaluation_design.csv", index=False)
heldout_unique.to_csv(OUT / "heldout_families.csv", index=False)
(OUT / "feature_audit.json").write_text(json.dumps(FEATURE_AUDIT, ensure_ascii=False, indent=2), encoding="utf-8")


## 2. Download and verify existing feature checkpoints

The existing 11 ZIP groups contain a complete, already-executed 5,200-prompt run. This notebook does not need an LLM or GPU. Files are **read only** from the original results folder.


In [ ]:
PROMPT_HASH_EXPECTED = '566d106415f80a059d8b6133ba804bd508f0c793052014f147644bfad32340d9'
assert PROMPT_HASH == PROMPT_HASH_EXPECTED, 'Stimulus bank changed; refuses to mix incompatible hidden states.'
N_PROMPTS = len(PROMPTS)
HIDDEN_SIZE = 896
FEATURE_KEY = 'L24_assistant_start'
REMOTE_CHECKPOINT_INTERVAL = 8
FEATURES = np.empty((N_PROMPTS, HIDDEN_SIZE), dtype=np.float32)
filled = np.zeros(N_PROMPTS,dtype=bool)
expected_groups=math.ceil(math.ceil(N_PROMPTS/BATCH_SIZE)/REMOTE_CHECKPOINT_INTERVAL)
assert expected_groups == 11
for group in tqdm(range(expected_groups), desc='Restoring verified checkpoint ZIPs'):
    remote_path=f'{SOURCE_PREFIX}/chunk_{group:03d}.zip'
    content=github_read(remote_path)
    assert content is not None, f'Missing completed source checkpoint: {remote_path}'
    with zipfile.ZipFile(io.BytesIO(content)) as archive:
        manifest=json.loads(archive.read('checkpoint_manifest.json'))
        assert manifest['prompt_hash'] == PROMPT_HASH
        assert manifest['n_prompts'] == N_PROMPTS
        assert manifest['model'] == 'Qwen/Qwen2.5-0.5B-Instruct'
        assert manifest['revision'] == '7ae557604adf67be50417f59c2c2f167def9a775'
        assert FEATURE_KEY in [f'L{layer}_{position}' for layer in manifest['layers'] for position in manifest['positions']]
        first_batch=group*REMOTE_CHECKPOINT_INTERVAL
        last_batch=min((group+1)*REMOTE_CHECKPOINT_INTERVAL, math.ceil(N_PROMPTS/BATCH_SIZE))
        expected_members={'checkpoint_manifest.json'}|{f'batch_{i:04d}.npz' for i in range(first_batch,last_batch)}
        assert set(archive.namelist()) == expected_members
        for batch_i in range(first_batch,last_batch):
            with np.load(io.BytesIO(archive.read(f'batch_{batch_i:04d}.npz'))) as batch:
                assert str(batch['prompt_hash'].item()) == PROMPT_HASH
                lo,hi=int(batch['start'].item()),int(batch['stop'].item())
                assert lo == batch_i*BATCH_SIZE and hi == min(lo+BATCH_SIZE,N_PROMPTS)
                assert not filled[lo:hi].any()
                array=batch[FEATURE_KEY].astype(np.float32)
                assert array.shape == (hi-lo,HIDDEN_SIZE) and np.isfinite(array).all()
                FEATURES[lo:hi]=array
                filled[lo:hi]=True
assert filled.all()
print(f'Loaded {N_PROMPTS} original hidden-state vectors, shape={FEATURES.shape}; no new model calls.')


## 3. Preregister an expanded comparator pool

To avoid trivially easy or meaningless comparisons, new pairs are limited to manually predefined broad semantic groups. We also retain all 20 original pair orientations as controls. Here *stable target concept* means that a particular X can be reliably distinguished from **at least two substantively different Ys**, not simply that one contrast has a perfect AUC. Reverse orientations of the same unordered pair do not provide independent confirmation.


In [ ]:
# Every name is from the frozen 40-concept stimulus bank; no external model is queried.
GROUPS = {
    'mammals': ['elephant','giraffe','cat','dog','lion','tiger','dolphin'],
    'other_animals': ['shark','eagle','penguin','butterfly','spider'],
    'produce': ['banana','apple','carrot','potato','lemon','orange'],
    'flowers': ['rose','sunflower'],
    'staple_foods': ['bread','rice'],
    'vehicles': ['bicycle','motorcycle','train','airplane','boat','submarine'],
    'tools': ['hammer','screwdriver','scissors','knife'],
    'time_instruments': ['clock','calendar'],
    'household_appliances': ['refrigerator','oven'],
    'optical_instruments': ['camera','telescope'],
    'musical_instruments': ['piano','violin']
}
assert set(sum(GROUPS.values(), [])) == set(TRAIN_FACTS)
assert len(sum(GROUPS.values(),[])) == 40

# A deterministic sampling rule ensures a fixed number of alternatives per X,
# rather than selecting comparators using the observed AUC.
comparisons=[]
for group, concepts in GROUPS.items():
    for i,x in enumerate(concepts):
        if len(concepts) >= 4:
            # Two alternatives chosen at deterministic positions around the group.
            choices=[concepts[(i+1)%len(concepts)], concepts[(i+2)%len(concepts)]]
        elif len(concepts) == 2:
            choices=[concepts[1-i]]
        else:
            raise AssertionError('Unexpected group size')
        for y in choices:
            comparisons.append({'x':x,'y':y,'group':group,'original_pair': PAIRS.get(x)==y})
# Include original X/Y contrasts if not generated by the fixed rotation.
seen={(item['x'],item['y']) for item in comparisons}
for x,y in PAIRS.items():
    if (x,y) not in seen:
        group=next(k for k,v in GROUPS.items() if x in v)
        comparisons.append({'x':x,'y':y,'group':group,'original_pair':True})
        seen.add((x,y))
PAIR_GRID=pd.DataFrame(comparisons).drop_duplicates(['x','y']).reset_index(drop=True)
PAIR_GRID['unordered_pair']=PAIR_GRID.apply(lambda r:'__'.join(sorted([r.x,r.y])), axis=1)
PAIR_GRID.to_csv(OUT/'candidate_pairs_preregistered.csv',index=False)
print('Directed candidate pairs:',len(PAIR_GRID))
print('Unique unordered contrasts:', PAIR_GRID['unordered_pair'].nunique())
print('Distinct X targets:',PAIR_GRID['x'].nunique())
print('Comparisons per target:')
display(PAIR_GRID.groupby('x').size().value_counts().sort_index())
display(PAIR_GRID)


## 4. Train a Probe on each contrast; evaluate independent description families

The wording and hidden states of each concept are **identical** to the prior run. Only the Y comparator and Probe training labels change. Classification AUC is computed after averaging the four wrappers of each of five unseen semantic-description families. Pair symmetry is checked explicitly; results are presented by *unordered* contrast for inferential counting.


In [ ]:
# Index the original neutral prompt bank by concept; the new X/Y pairs
# are constructed using the cached vectors without requiring new prompts.
TRAIN_ROWS={concept:train_df[train_df.concept==concept].copy().reset_index(drop=True) for concept in TRAIN_FACTS}
EVAL_ROWS={concept:eval_df[(eval_df.concept==concept)&(eval_df.condition=='neutral')].copy().reset_index(drop=True) for concept in TRAIN_FACTS}
for concept in TRAIN_FACTS:
    assert len(TRAIN_ROWS[concept])==30
    assert len(EVAL_ROWS[concept])==20
    assert set(TRAIN_ROWS[concept]['family_index'])==set(range(6))
    assert set(EVAL_ROWS[concept]['family_index'])==set(range(5))

def indices(frame):
    return np.fromiter((PROMPT_TO_INDEX[p] for p in frame['prompt']), dtype=int,count=len(frame))

def probe():
    return LogisticRegression(C=C,solver='liblinear',random_state=SEED,max_iter=2000)

def evaluate_auc(y_scores, x_scores):
    # Scores are 5 held-out semantic families x 4 wrappers for each concept.
    assert len(y_scores)==len(x_scores)==20
    y_fam=np.mean(y_scores.reshape(5,4),axis=1)
    x_fam=np.mean(x_scores.reshape(5,4),axis=1)
    labels=np.r_[np.zeros(5),np.ones(5)]
    return float(roc_auc_score(labels,np.r_[y_fam,x_fam])),x_fam,y_fam

# Screening uses six training-family folds, like the original paper protocol.
# Unique *unordered* contrasts are analyzed only once, even if both orientations
# were in the prospective comparison table.
UNIQUE_PAIRS=PAIR_GRID.drop_duplicates('unordered_pair').copy().reset_index(drop=True)
fold_records=[]
for row in tqdm(UNIQUE_PAIRS.itertuples(),total=len(UNIQUE_PAIRS),desc='Screening unique contrasts'):
    a,b=row.x,row.y
    for fold in range(6):
        atr=TRAIN_ROWS[a][TRAIN_ROWS[a].family_index!=fold]
        btr=TRAIN_ROWS[b][TRAIN_ROWS[b].family_index!=fold]
        train_idx=np.r_[indices(atr),indices(btr)]
        train_labels=np.r_[np.ones(len(atr)),np.zeros(len(btr))]
        scaler=StandardScaler().fit(FEATURES[train_idx])
        X=scaler.transform(FEATURES[train_idx])
        fitted=probe().fit(X,train_labels)
        a_scores=fitted.decision_function(scaler.transform(FEATURES[indices(EVAL_ROWS[a])]))
        b_scores=fitted.decision_function(scaler.transform(FEATURES[indices(EVAL_ROWS[b])]))
        auc,a_family,b_family=evaluate_auc(b_scores,a_scores)
        leave_one=[float(roc_auc_score(np.r_[np.zeros(4),np.ones(4)],np.r_[np.delete(b_family,k),np.delete(a_family,k)])) for k in range(5)]
        fold_records.append({'unordered_pair':row.unordered_pair,'a':a,'b':b,'group':row.group,
            'fold':fold,'neutral_auc':auc,'min_family_drop_auc':min(leave_one),
            'mean_family_drop_auc':float(np.mean(leave_one))})
fold_df=pd.DataFrame(fold_records)
assert len(fold_df)==len(UNIQUE_PAIRS)*6
fold_df.to_csv(OUT/'screening_fold_auc.csv',index=False)
contrast_df=fold_df.groupby(['unordered_pair','a','b','group'],as_index=False).agg(
    mean_auc=('neutral_auc','mean'),min_auc=('neutral_auc','min'),median_auc=('neutral_auc','median'),
    folds_auc_ge_075=('neutral_auc',lambda s:int((s>=0.75).sum())),
    folds_auc_ge_080=('neutral_auc',lambda s:int((s>=0.8).sum())),
    median_worst_family_drop_auc=('min_family_drop_auc','median'))
contrast_df['screen_pass']=((contrast_df.folds_auc_ge_075>=5)&(contrast_df.median_worst_family_drop_auc>=0.7))
contrast_df=contrast_df.sort_values(['screen_pass','mean_auc'],ascending=False)
contrast_df.to_csv(OUT/'screening_unique_contrasts.csv',index=False)
print('Unique contrasts meeting AUC/family-drop screening:',int(contrast_df.screen_pass.sum()),'/',len(contrast_df))
display(contrast_df.head(35))


## 5. Which **distinct targets X** have at least two good comparators?

A contrast is symmetric in its underlying class separation. Target-level counting is therefore performed **only once per distinct X**. Screening success is not a guarantee of significance; the separate exact-shuffle check follows. The same original descriptions are reused across all comparisons, so these are correlated exploratory results.


In [ ]:
pair_scores=dict(zip(contrast_df.unordered_pair,contrast_df.mean_auc))
pair_pass=dict(zip(contrast_df.unordered_pair,contrast_df.screen_pass))
target_rows=[]
for x,frame in PAIR_GRID.groupby('x'):
    good=frame[frame.unordered_pair.map(pair_pass).fillna(False)]
    target_rows.append({'target':x,'group':frame.group.iloc[0],
        'comparators_tested':int(len(frame)),
        'comparators_screen_pass':int(len(good)),
        'passed_comparators':', '.join(good.y.tolist()),
        'best_comparator':(good.iloc[0].y if len(good) else None),
        'best_auc':float(max((pair_scores[k] for k in good.unordered_pair),default=float('nan'))),
        'two_comparators_screen_pass':bool(len(good)>=2)})
target_df=pd.DataFrame(target_rows).sort_values(['two_comparators_screen_pass','comparators_screen_pass','best_auc'],ascending=False)
target_df.to_csv(OUT/'target_concept_screening.csv',index=False)
print('Distinct X concepts passing with ≥2 comparator concepts:',target_df.two_comparators_screen_pass.sum(),'/40')
display(target_df)


## 6. Exact label-shuffle controls for screening candidates

This step runs **252 balanced family-label assignments × 6 training-family folds** for each unique contrast passing the predeclared AUC/family-drop screen. It may take considerably longer than the first screen. Set `RUN_EXACT_SHUFFLE=False` only for a fast preliminary screen; the exported status explicitly records that such an output is incomplete.


In [ ]:
RUN_EXACT_SHUFFLE = True
CHECK_SHUFFLE_ONLY_SCREEN_PASSES = True  # Avoid evaluating unpromising contrasts; exploratory screening, not family-wise hypothesis test.
SHUFFLE_PAIR_LIST = contrast_df[contrast_df.screen_pass].copy() if CHECK_SHUFFLE_ONLY_SCREEN_PASSES else contrast_df.copy()
print('Unique contrasts to permutation-test:',len(SHUFFLE_PAIR_LIST))

shuf_records=[]
if RUN_EXACT_SHUFFLE:
    with threadpool_limits(limits=2):
        for r in tqdm(SHUFFLE_PAIR_LIST.itertuples(),total=len(SHUFFLE_PAIR_LIST),desc='Exact family-level permutations'):
            a,b=r.a,r.b
            for fold in range(6):
                atr=TRAIN_ROWS[a][TRAIN_ROWS[a].family_index!=fold]
                btr=TRAIN_ROWS[b][TRAIN_ROWS[b].family_index!=fold]
                tr=pd.concat([atr,btr],ignore_index=True)
                family_ids=sorted(tr.family.unique())
                assert len(family_ids)==10
                XX=FEATURES[indices(tr)]
                scaler=StandardScaler().fit(XX)
                XX=scaler.transform(XX)
                ai=scaler.transform(FEATURES[indices(EVAL_ROWS[a])])
                bi=scaler.transform(FEATURES[indices(EVAL_ROWS[b])])
                actual_auc=float(fold_df[(fold_df.unordered_pair==r.unordered_pair)&(fold_df.fold==fold)].neutral_auc.iloc[0])
                all_auc=[]
                for perm_id, selected in enumerate(itertools.combinations(family_ids,5)):
                    positive=set(selected)
                    labels=np.fromiter((int(f in positive) for f in tr.family),dtype=int,count=len(tr))
                    classifier=probe().fit(XX,labels)
                    a_scores=classifier.decision_function(ai)
                    b_scores=classifier.decision_function(bi)
                    null_auc,_,_=evaluate_auc(b_scores,a_scores)
                    all_auc.append(null_auc)
                assert len(all_auc)==252
                tail=float(np.mean(np.array(all_auc)>=actual_auc-1e-12))
                shuf_records.append({'unordered_pair':r.unordered_pair,'fold':fold,'actual_auc':actual_auc,
                    'exact_upper_tail_proportion':tail,'num_balanced_permutations':252,
                    'null_mean_auc':float(np.mean(all_auc)),'null_max_auc':float(np.max(all_auc))})
    assert len(shuf_records)==len(SHUFFLE_PAIR_LIST)*6
shuf_df=pd.DataFrame(shuf_records,columns=['unordered_pair','fold','actual_auc','exact_upper_tail_proportion','num_balanced_permutations','null_mean_auc','null_max_auc'])
shuf_df.to_csv(OUT/'exact_shuffles_by_contrast_fold.csv',index=False)
if RUN_EXACT_SHUFFLE:
    significance=shuf_df.groupby('unordered_pair',as_index=False).agg(
       folds_shuffle_p_le_005=('exact_upper_tail_proportion',lambda s:int((s<=0.05).sum())),
       median_shuffle_p=('exact_upper_tail_proportion','median'))
    contrast_df=contrast_df.merge(significance,on='unordered_pair',how='left')
    contrast_df['exploratory_stable_pair']=(contrast_df['screen_pass']&(contrast_df['folds_shuffle_p_le_005'].fillna(0)>=5))
else:
    contrast_df['folds_shuffle_p_le_005']=np.nan
    contrast_df['median_shuffle_p']=np.nan
    contrast_df['exploratory_stable_pair']=False
contrast_df.to_csv(OUT/'contrast_results_with_shuffles.csv',index=False)
print('Exploratory stable unique contrasts:',int(contrast_df.exploratory_stable_pair.sum()))

target_stable=[]
stable_map=dict(zip(contrast_df.unordered_pair,contrast_df.exploratory_stable_pair))
for x,sub in PAIR_GRID.groupby('x'):
    valid=sub[sub.unordered_pair.map(stable_map).fillna(False)]
    target_stable.append({'target':x,'stable_distinct_comparators':len(valid),
        'comparators':', '.join(valid.y.tolist()),
        'two_comparators_stable':len(valid)>=2})
target_stable_df=pd.DataFrame(target_stable).sort_values(['two_comparators_stable','stable_distinct_comparators'],ascending=False)
target_stable_df.to_csv(OUT/'stable_target_candidates.csv',index=False)
print('Distinct X concepts with ≥2 exploratory-stable Y comparators:',int(target_stable_df.two_comparators_stable.sum()),'/40')
display(target_stable_df)


## 7. Publish experiment records and summaries to GitHub

A new run-specific results folder is used, leaving the completed 20-pair experiment untouched. Re-running the analysis updates only the files belonging to this new run. Results are **candidate screening**. The next step for the final 20 targets is a separate held-out stimulus set to address selection bias and evaluate semantic-distance manipulation.


In [ ]:
manifest={
  'status':'COMPLETED_SCREEN_AND_SHUFFLE' if RUN_EXACT_SHUFFLE else 'SCREEN_ONLY_NO_SHUFFLE',
  'created_utc':datetime.now(timezone.utc).isoformat(),
  'source_experiment':'1110_qwen_probe_20pairs/results/20261110_qwen20pairs_v1',
  'source_prompt_sha256':PROMPT_HASH,
  'source_model':'Qwen/Qwen2.5-0.5B-Instruct',
  'source_model_revision':'7ae557604adf67be50417f59c2c2f167def9a775',
  'source_unique_prompts':N_PROMPTS,
  'model_forward_calls_in_this_run':0,
  'readout':'L24_assistant_start',
  'instruction_condition':'neutral',
  'unique_target_concepts':PAIR_GRID.x.nunique(),
  'directed_candidate_pairs':len(PAIR_GRID),
  'unique_unordered_contrasts':len(UNIQUE_PAIRS),
  'unique_contrasts_permutation_tested':len(SHUFFLE_PAIR_LIST) if RUN_EXACT_SHUFFLE else 0,
  'shuffle_fits':len(shuf_records)*252 if RUN_EXACT_SHUFFLE else 0,
  'minimum_target_comparators_for_stability':2,
  'target_stability_status':'exploratory_only',
  'caveats':[
      'No new concepts or descriptions were collected; hidden features of the old 40 concepts were reused.',
      'Candidate contrasts share the same training and evaluation descriptions; they are not independent replications.',
      'Selecting pairs based on held-out performance requires independent fresh-text confirmation.',
      'Comparing distinct, easier Y concepts may inflate AUC while weakening relevance to suppression-distance research.',
      'X/Y reverse directions of the same unordered pair do not constitute independent observations.',
      'Exact shuffles are applied only to candidates passing the screen when CHECK_SHUFFLE_ONLY_SCREEN_PASSES=True.',
      'No multiple comparisons correction; per-contrast shuffle proportions should not be construed as confirmatory p-values.',
      'The Probe classifies relative X-versus-Y identity; it does not measure absolute concept activation or causality.'
  ]
}
(OUT/'manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf8')
summary=(f'# Comparator-pair screening of 40 concepts\n\n'
  f'- **{len(PAIR_GRID)} directed candidate contrasts** / **{len(UNIQUE_PAIRS)} unique unordered contrasts**\n'
  f'- **{int(contrast_df.exploratory_stable_pair.sum())} unique contrasts** met exploratory stability criterion.\n'
  f'- **{int(target_stable_df.two_comparators_stable.sum())} distinct targets** have at least two stable alternative Ys.\n'
  f'- Input: previously generated 5200 Qwen hidden-state records (no new model inference).\n'
  f'- Exact family-label permutations completed: {len(shuf_records)*252:,} fits.\n\n'
  '## Candidate targets by number of stable comparators\n\n'
  +target_stable_df.to_markdown(index=False)+'\n\n'
  '## Pair-level screening results\n\n'
  +contrast_df.to_markdown(index=False)+'\n\n'
  '**Interpretation:** screening only. The selected 20 concept targets require new independently authored descriptions and matched semantic-distance cues to confirm out-of-sample performance.\n')
(OUT/'analysis_summary.md').write_text(summary,encoding='utf8')
# chart
plot=contrast_df.sort_values('mean_auc',ascending=False).head(30).iloc[::-1]
fig,ax=plt.subplots(figsize=(10, max(7,len(plot)*0.28)))
ax.barh(plot.unordered_pair,plot.mean_auc)
ax.axvline(0.5,color='gray',linestyle='--',linewidth=0.8)
ax.set_xlim(0,1.02)
ax.set_xlabel('Mean family-level ROC-AUC (6 training folds)')
ax.set_title('Top 30 exploratory X/Y contrasts')
fig.tight_layout()
fig.savefig(OUT/'screened_contrasts_top30.png',dpi=165)
plt.show()

zip_path=OUT/'qwen_probe_comparator_expansion_results.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUT.iterdir()):
        if path.is_file() and path!=zip_path:
            z.write(path,arcname=path.name)
for path in sorted(OUT.iterdir()):
    if path.is_file() and path!=zip_path:
        github_publish(f'{DEST_PREFIX}/{path.name}',path.read_bytes())
github_publish(f'{DEST_PREFIX}/{zip_path.name}',zip_path.read_bytes())
print('Published:',f'https://github.com/{GITHUB_REPOSITORY}/tree/main/{DEST_PREFIX}')
